# Najafi et al. (2020) on DANDI:000016, notebook 5: mixed and pseudo populations, outcome decoding

**Paper.** Najafi F, Elsayed GF, Cao R, Pnevmatikakis E, Latham PE, Cunningham JP, Churchland AK (2020).
Excitatory and inhibitory subnetworks are equally selective during decision-making and emerge
simultaneously during learning. *Neuron* 105, 165-179. [doi:10.1016/j.neuron.2019.09.045](https://doi.org/10.1016/j.neuron.2019.09.045)

**Dataset.** [DANDI:000016](https://dandiarchive.org/dandiset/000016/0.260915.2309), version
0.260915.2309 (doi:10.48324/dandi.000016/0.260915.2309), streamed from the archive. This is the only
published version (2026-09-15); the paper cites no DANDI version (it names the NWB deposit at the
CSHL repository, doi:10.14224/1.37693, from which the dandiset is derived), and the draft holds
the same 135 files.

**Claims.** `claims/000016/10.1016-j.neuron.2019.09.045.yaml`, written from the accepted author
manuscript in PubMed Central (PMC6952547) with its supplement, and the paper's code
(github.com/farznaj/imaging_decisionMaking_exc_inh at commit b7352acd). The claims file decides what
is tested and how; this notebook does not change a claim.

Claims tested here (the full table is in notebook 1):

| Claim | Finding | Notebook | Status |
|---|---|---|---|
| `fig3hi-mixed-no-advantage` | Mixed E and I population no better than E | 05 | tested |
| `figs6c-outcome-decoding` | Outcome decoding reaches 80% | 05 | tested |
| `fig7-noise-correlations-limit-decoding` | Pseudo populations decode better | 05 | tested |

For run time, fig3hi uses 10 random subsamples per session where the claims file has 50, and the
outcome decoders of figs6c use 5 excitatory subsamples; deviations, recorded.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import najafi as nj
import sessions as S

pd.set_option("display.width", 200)
MICE = [1, 2, 3, 4]
A = nj.asset_urls()
print(f"DANDI:{nj.DANDISET_ID} version {nj.DANDISET_VERSION}: {len(A)} NWB files")
items = [(i, p, u) for i, (p, u) in enumerate(zip(A.path, A.url))]

R = nj.map_sessions(S.decode_variants, items, workers=4)
RO = nj.map_sessions(S.decode_outcome, items, workers=4)
OK = [r for r in R if r["ok"]]
print(len(R), "sessions loaded;", len(OK), "used for fig3hi and fig7;", sum(r["ok"] for r in RO), "for figs6c")

## Data check

In [ ]:
d = pd.DataFrame([dict(mouse=r["mouse"], ok=r["ok"], inh=r["n_inh"], exc=r["n_exc"], all=r["n_all"], left=r["n_left"], right=r["n_right"]) for r in R])
print(d.groupby("mouse").agg(sessions=("ok", "size"), used=("ok", "sum"), inh_mean=("inh", "mean"), exc_mean=("exc", "mean"),
                             all_mean=("all", "mean"), correct_high=("left", "mean"), correct_low=("right", "mean")).round(1).to_string())
print("Sessions without 2n excitatory neurons for fig3hi:", sum(1 for r in OK if len(r["mixed"]) == 0))
o = pd.DataFrame([dict(mouse=r["mouse"], ok=r["ok"], cor=r["n_correct"], inc=r["n_incorrect"], cell=r["n_min_cell"],
                       first=r["bin_times"][0], last=r["bin_times"][-1], b400=r.get("bin400")) for r in RO])
print(o.groupby("mouse").agg(sessions=("ok", "size"), used=("ok", "sum"), correct=("cor", "mean"), incorrect=("inc", "mean"),
                             min_cell=("cell", "min"), last_bin_ms_min=("last", "min")).round(1).to_string())
print("Outcome-aligned frame periods equal for reward and incorrect alignments:", all(abs(r["rate_reward"] - r["rate_incorr"]) < 1e-9 for r in RO))
print("Sessions whose outcome-aligned traces reach the bin containing +400 ms:", int(o.b400.notna().sum()), "of", len(o))

## fig3hi-mixed-no-advantage

> A population of half excitatory and half inhibitory neurons decodes choice with accuracy similar
> to that of an excitatory population of the same size, so a mixed population offers no major
> advantage.

Criterion: mixed population = all n inhibitory neurons plus n random excitatory neurons, control = 2n
excitatory neurons, over subsamples; correct trials, 0-97 ms before the choice; unpaired two-tailed
t-test across sessions per mouse at p < 0.05, a claim of absence. Reproduced when no mouse shows a
difference in either direction.

In [ ]:
rows = []
for m in MICE:
    rr = [r for r in OK if r["mouse"] == m and len(r["mixed"])]
    mx = np.array([r["mixed"].mean() for r in rr]); ex = np.array([r["exc2n"].mean() for r in rr])
    tt = nj.ttest(mx, ex)
    rows.append(dict(mouse=m, sessions=len(rr), mixed=mx.mean(), exc_2n=ex.mean(), mixed_minus_exc=mx.mean() - ex.mean(), t=tt[0], p=tt[1]))
T = pd.DataFrame(rows)
print(T.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Mice with p < 0.05:", T.mouse[T.p < 0.05].tolist())

**Result: (to be written)**

## fig7-noise-correlations-limit-decoding

> Removing noise correlations by building pseudo populations increases choice decoding accuracy of
> the full population in every mouse.

Criterion: all neurons at 0-97 ms before the choice; pseudo populations shuffle trial order
independently for each neuron within each choice; the change is pseudo minus real accuracy at the
largest ensemble. The largest ensemble holds every neuron, so the order in which neurons are added
does not change it, and only the full population is decoded here. Trials (main) correct trials and
(second reading) all trials with a choice. Reproduced when the mean change is positive in every
mouse and significant by the two-tailed t-test across sessions, read as (a) a one-sample test of
the changes and (b) an unpaired test of pseudo against real accuracy.

In [ ]:
rows = []
for ts in ("correct", "all"):
    for m in MICE:
        rr = [r for r in OK if r["mouse"] == m and ts in r["pseudo"]]
        real = np.array([r["pseudo"][ts]["real"] for r in rr]); ps = np.array([r["pseudo"][ts]["pseudo"] for r in rr])
        ch = ps - real
        rows.append(dict(trials=ts, mouse=m, sessions=len(rr), real=real.mean(), pseudo=ps.mean(), change=ch.mean(),
                         p_a_one_sample=stats.ttest_1samp(ch, 0).pvalue, p_b_unpaired=nj.ttest(ps, real)[1]))
T = pd.DataFrame(rows)
print(T.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
for ts in ("correct", "all"):
    s_ = T[T.trials == ts]
    print(f"{ts} trials: positive in every mouse {bool((s_.change > 0).all())}; significant (a) {bool((s_.p_a_one_sample < 0.05).all())}, (b) {bool((s_.p_b_unpaired < 0.05).all())}")

**Result: (to be written)**

## figs6c-outcome-decoding

> Population activity is strongly selective for trial outcome, reaching about 80% accuracy (median
> across mice) about 400 ms after the confirmation lick, with inhibitory neurons slightly more
> accurate than excitatory neurons after reward.

Criterion: outcome-aligned trials (aligned to the confirmation lick): the files give the correct
trials aligned to the reward (`dFoF_rewardAl`) and the incorrect trials aligned to their second
commit lick (`dFoF_commitIncorrAl`), combined on the frames both windows share. Equal numbers of
left and right choice trials for each outcome. The all-neuron accuracy at the bin containing 400 ms
after the confirmation lick, mean over sessions per mouse, median over mice, within 80 +/- 8%.
Inhibitory above subsampled excitatory in every mouse, in the mean over bins after the confirmation
lick, the window ending (a) 500 ms after it (bins starting before 500 ms) and (b) at the end of the
outcome-aligned traces; the unpaired t-test across sessions is reported. Reproduced when both hold.

In [ ]:
OKo = [r for r in RO if r["ok"]]
rows = []
for m in MICE:
    rr = [r for r in OKo if r["mouse"] == m]
    a4 = np.array([r["acc_all"][r["bin400"]] for r in rr if r["bin400"] is not None and r["bin400"] in r["acc_all"]])
    s4 = np.array([r["shfl_all"][r["bin400"]] for r in rr if r["bin400"] is not None and r["bin400"] in r["shfl_all"]])
    row = dict(mouse=m, sessions=len(rr), sessions_400=len(a4), all_400=a4.mean() if len(a4) else np.nan,
               shfl_400=s4.mean() if len(s4) else np.nan)
    for w, lim in (("a", 500), ("b", np.inf)):
        ie = [(np.mean([r["acc_inh"][b] for b in r["acc_inh"] if r["bin_times"][b] < lim]),
               np.mean([r["acc_exc"][b] for b in r["acc_exc"] if r["bin_times"][b] < lim])) for r in rr]
        ie = np.array(ie)
        row[f"inh_{w}"], row[f"exc_{w}"] = ie[:, 0].mean(), ie[:, 1].mean()
        row[f"p_{w}"] = nj.ttest(ie[:, 0], ie[:, 1])[1]
    rows.append(row)
T = pd.DataFrame(rows)
print(T.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Median over mice of the all-neuron accuracy at +400 ms: %.1f%% (expected 80 +/- 8)" % np.nanmedian(T.all_400))
print("Inhibitory above excitatory in every mouse: window (a)", bool((T.inh_a > T.exc_a).all()), ", window (b)", bool((T.inh_b > T.exc_b).all()))
print("Post-lick bins per session: %d to %d; last bin starts %.0f to %.0f ms" % (
    min(len(r["acc_inh"]) for r in OKo), max(len(r["acc_inh"]) for r in OKo),
    min(max(r["bin_times"][b] for b in r["acc_inh"]) for r in OKo), max(max(r["bin_times"][b] for b in r["acc_inh"]) for r in OKo)))

**Result: (to be written)**

## Summary

(to be written)